# Notebook 04: Violation-Level Feature Engineering

## Purpose

Build the final case-level modeling table for unresolved violation prioritization.

- **One row per violation case**.
- **Target:** `is_unresolved` (1 if `status == Open`, 0 if `status == Closed`).
- **Leakage control:** target-derived fields such as `status`, `status_clean`, `status_dttm`, `status_year`, `status_month`, and `case_no` are documented but excluded from model inputs.

The outputs are `data/processed/violation_level_model_features.csv` and `data/processed/violation_level_feature_columns.json`, which are consumed by Notebook 05.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'

violations_path = PROCESSED_DIR / 'violations_clean.csv'
structural_path = EXTERNAL_DIR / 'neighborhood_structural_features.csv'
rentsmart_context_path = EXTERNAL_DIR / 'rentsmart_neighborhood_context.csv'

if not violations_path.exists():
    raise FileNotFoundError(f'Missing cleaned violations: {violations_path}. Run Notebook 02 first.')

violations = pd.read_csv(violations_path, low_memory=False)
print(f'Loaded {len(violations):,} cleaned violation rows.')
violations.head(3)

Loaded 16,914 cleaned violation rows.


,case_no,status,status_dttm,year,month,day_of_week,code,description,violation_city,neighborhood,...,violation_stno,violation_sthigh,violation_street,violation_suffix,violation_zip,ward,latitude,longitude,has_coords,has_neighborhood
0,V897990,Open,2026-03-20 15:27:46,2026.0,3.0,Friday,107.4,Failed to comply w permit term,Dorchester,Dorchester,...,14,NaN,Elder,ST,02125,07,42.32023,-71.06334,True,True
1,V897956,Open,2026-03-20 10:37:20,2026.0,3.0,Friday,102.8,Maintenance,Dorchester,Dorchester,...,111,NaN,Quincy,ST,02121,13,42.31383,-71.07762,True,True
2,V897950,Open,2026-03-20 10:22:23,2026.0,3.0,Friday,102.8,Maintenance,Roxbury,Roxbury,...,610,610A,Shawmut,AV,02118,09,42.33555,-71.08033,True,True


## 1. Define Target

The target is based on the observed case status in the data snapshot. We exclude partial 2026 records from the modeling table so the model is less affected by incomplete current-year reporting.

In [2]:
df = violations.copy()
df['status_clean'] = df['status'].astype(str).str.strip().str.lower()
df = df[df['status_clean'].isin(['open', 'closed'])].copy()
df['status_year'] = pd.to_numeric(df.get('year'), errors='coerce')
df['status_month'] = pd.to_numeric(df.get('month'), errors='coerce')
df = df[df['status_year'].fillna(0) < 2026].copy()
df['is_unresolved'] = (df['status_clean'] == 'open').astype(int)

summary = pd.DataFrame({
    'metric': ['model_rows', 'unresolved_rows', 'closed_rows', 'unresolved_rate_pct', 'target_definition', 'excluded_records'],
    'value': [
        len(df),
        int(df['is_unresolved'].sum()),
        int((df['is_unresolved'] == 0).sum()),
        round(df['is_unresolved'].mean() * 100, 2),
        '1 if status == Open, 0 if status == Closed; 2026 partial records excluded from training table',
        len(violations) - len(df),
    ],
})
summary.to_csv(PROCESSED_DIR / 'violation_level_target_summary.csv', index=False)
summary

,metric,value
0,model_rows,16760
1,unresolved_rows,794
2,closed_rows,15966
3,unresolved_rate_pct,4.74
4,target_definition,"1 if status == Open, 0 if status == Closed; 20..."
5,excluded_records,154


## 2. Extract Case-Level Features

The model uses violation type, location, missingness flags, and neighborhood context. Neighborhood context combines population, housing-stock summaries, and aggregated RentSmart current-case property context. We do not directly use owner names or historical complaint JSON fields because they can overfit or leak target-related information. The model also does not use the status fields that define the label.

In [ ]:
def clean_zip(value):
    if pd.isna(value):
        return 'Missing'
    digits = ''.join(ch for ch in str(value) if ch.isdigit())
    return digits[:5] if len(digits) >= 5 else 'Missing'


def group_description(value):
    text = str(value).lower()
    if 'maintenance' in text or 'condition' in text:
        return 'maintenance_condition'
    if 'egress' in text or 'exit' in text:
        return 'egress_exit'
    if 'electrical' in text or 'wiring' in text:
        return 'electrical'
    if 'plumbing' in text or 'water' in text:
        return 'plumbing_water'
    if 'heat' in text or 'heating' in text:
        return 'heating'
    if 'trash' in text or 'garbage' in text or 'sanitary' in text:
        return 'sanitation'
    if 'certificate' in text or 'testing' in text:
        return 'testing_certification'
    return 'other'


df['code_key'] = df['code'].astype(str).str.strip().replace({'nan': 'Missing', '': 'Missing'})
code_counts = df['code_key'].value_counts()
rare_codes = set(code_counts[code_counts < 20].index)
df['code_grouped'] = np.where(df['code_key'].isin(rare_codes), 'Rare code', df['code_key'])
df['description_group'] = df['description'].map(group_description)
df['zip5'] = df['violation_zip'].map(clean_zip)
df['ward_clean'] = df['ward'].fillna('Missing').astype(str).str.strip().replace('', 'Missing')
df['violation_suffix_clean'] = df['violation_suffix'].fillna('Missing').astype(str).str.strip().replace('', 'Missing')
df['has_coords'] = df['latitude'].notna() & df['longitude'].notna()
df['has_neighborhood'] = df['neighborhood'].notna() & (df['neighborhood'].astype(str).str.strip() != '')
df['neighborhood'] = df['neighborhood'].fillna('Missing').astype(str)
df['neighborhood_assignment_method'] = df['neighborhood_assignment_method'].fillna('Missing').astype(str)

if structural_path.exists():
    structural = pd.read_csv(structural_path)
    df = df.merge(structural, on='neighborhood', how='left')
else:
    print(f'Warning: missing structural features: {structural_path}')

if rentsmart_context_path.exists():
    rentsmart_context = pd.read_csv(rentsmart_context_path)
    df = df.merge(rentsmart_context, on='neighborhood', how='left')
else:
    print(f'Warning: missing RentSmart context features: {rentsmart_context_path}')

numeric_features = [
    'latitude', 'longitude', 'population_2025', 'property_count', 'median_building_age',
    'share_pre_1940', 'share_pre_1900', 'share_remodeled', 'median_total_value',
    'median_living_area', 'median_land_sf', 'share_condo', 'share_multifamily',
    'properties_per_1000_residents',
    'rentsmart_case_count', 'rentsmart_median_building_age', 'rentsmart_share_pre_1940',
    'rentsmart_share_pre_1900', 'rentsmart_share_remodeled', 'rentsmart_share_condo',
    'rentsmart_share_multifamily',
]
categorical_features = [
    'code_grouped', 'description_group', 'neighborhood', 'neighborhood_assignment_method',
    'zip5', 'ward_clean', 'violation_suffix_clean', 'has_coords', 'has_neighborhood',
]

numeric_features = [col for col in numeric_features if col in df.columns]
categorical_features = [col for col in categorical_features if col in df.columns]

keep_cols = [
    'case_no', 'status', 'status_dttm', 'year', 'month', 'day_of_week', 'code', 'description',
    'violation_city', 'neighborhood', 'neighborhood_assignment_method', 'violation_stno',
    'violation_sthigh', 'violation_street', 'violation_suffix', 'violation_zip', 'ward',
    'latitude', 'longitude', 'has_coords', 'has_neighborhood', 'status_year', 'status_month',
    'status_clean', 'is_unresolved', 'code_key', 'description_group', 'zip5', 'ward_clean',
    'violation_suffix_clean', 'code_grouped',
] + [col for col in numeric_features if col not in ['latitude', 'longitude']]
keep_cols = [col for col in keep_cols if col in df.columns]

model_df = df[keep_cols].copy()
features_path = PROCESSED_DIR / 'violation_level_model_features.csv'
metadata_path = PROCESSED_DIR / 'violation_level_feature_columns.json'
model_df.to_csv(features_path, index=False)

metadata = {
    'target': 'is_unresolved',
    'target_definition': '1 if status == Open, 0 if status == Closed; 2026 partial records excluded',
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'excluded_from_features': ['status', 'status_clean', 'status_dttm', 'status_year', 'status_month', 'case_no'],
    'target_strategy': 'status_open_closed',
}
metadata_path.write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))
print(f'Saved -> {features_path}')
print(f'Saved -> {metadata_path}')

SyntaxError: unterminated f-string literal (detected at line 85) (829742337.py, line 85)

## Summary

Notebook 04 writes the final violation-level feature table and feature metadata. Notebook 05 trains the classification models and compares them against simple baselines.